<a href="https://colab.research.google.com/github/Mehrun-c/GlobusSoft/blob/main/training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import os
import cv2
import numpy as np
import pickle
from sklearn.datasets import fetch_lfw_people
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

In [20]:
print("Loading LFW dataset (min 30 images per person)...")
lfw = fetch_lfw_people(min_faces_per_person=30, resize=0.5, color=False)

X_raw = lfw.images          # shape: (N, H, W)
y     = lfw.target          # integer labels
names = lfw.target_names    # string names

print(f"  Dataset: {X_raw.shape[0]} images | {len(names)} identities")
print(f"  Image size: {X_raw.shape[1]}x{X_raw.shape[2]} px")

Loading LFW dataset (min 30 images per person)...
  Dataset: 2370 images | 34 identities
  Image size: 62x47 px


In [21]:
def extract_hog_features(images: np.ndarray) -> np.ndarray:
    """
    Compute HOG (Histogram of Oriented Gradients) descriptor for each image.
    No external heavy library needed – uses only OpenCV.
    """
    # Define target HOG window size that satisfies the constraints
    # Current image size is (height=62, width=47)
    # HOG requirements: (winSize.width - blockSize.width) % blockStride.width == 0
    #                   (winSize.height - blockSize.height) % blockStride.height == 0
    # With blockSize=(8,8) and blockStride=(4,4):
    # For width: (W - 8) % 4 == 0. Smallest W >= 47 satisfying this is W=48. (48-8)%4 = 40%4 = 0
    # For height: (H - 8) % 4 == 0. Smallest H >= 62 satisfying this is H=64. (64-8)%4 = 56%4 = 0
    HOG_WIN_SIZE_W = 48
    HOG_WIN_SIZE_H = 64

    hog = cv2.HOGDescriptor(
        _winSize   =(HOG_WIN_SIZE_W, HOG_WIN_SIZE_H),   # (W, H)
        _blockSize =(8, 8),
        _blockStride=(4, 4),
        _cellSize  =(4, 4),
        _nbins     =9
    )
    features = []
    for img in images:
        # Resize image to the HOG_WIN_SIZE_H x HOG_WIN_SIZE_W
        img_resized = cv2.resize(img, (HOG_WIN_SIZE_W, HOG_WIN_SIZE_H))

        # Convert float [0,1] → uint8 [0,255] if necessary
        if img_resized.max() <= 1.0:
            img_resized = (img_resized * 255).astype(np.uint8)
        else:
            img_resized = img_resized.astype(np.uint8)
        feat = hog.compute(img_resized).flatten()
        features.append(feat)
    return np.array(features)

print("\nExtracting HOG features...")
X = extract_hog_features(X_raw)
print(f"  Feature vector size: {X.shape[1]}")


Extracting HOG features...
  Feature vector size: 5940


In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"\nTrain samples: {len(X_train)} | Test samples: {len(X_test)}")


Train samples: 1896 | Test samples: 474


In [17]:
print("\nTraining SVM model...")
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm",    SVC(kernel="rbf", C=5.0, gamma="scale",
                   probability=True, random_state=42))
])
pipeline.fit(X_train, y_train)


Training SVM model...


Pipeline(steps=[('scaler', StandardScaler()),
                ('svm', SVC(C=5.0, probability=True, random_state=42))])

In [18]:
model_data = {
    "pipeline"    : pipeline,
    "target_names": names,
    "image_shape" : X_raw.shape[1:],   # (H, W)
}

model_path = "face_auth_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(model_data, f)

print(f"\nModel saved → {model_path}")
print("Training complete!")



Model saved → face_auth_model.pkl
Training complete!
